In [ ]:
!pip install -q openai-whisper opencc-python-reimplemented
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*"):
    print(path)

In [ ]:
from pathlib import Path
import json
import shutil
import torch
import whisper
from IPython.display import display, Audio, FileLink
import os
from opencc import OpenCC

device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("small", device=device)
# "tiny", "base", "small", "medium", "large"

In [ ]:
AUDIO_DIR = Path(
    "/kaggle/input/datasets/snnguyn1234/listen-a-minute/listen_a_minute/"
)

# Find every MP3 file in the Kaggle dataset folder
audio_files = sorted(AUDIO_DIR.glob("*.mp3"))

output_root = Path("/kaggle/working/whisper_output")
output_root.mkdir(parents=True, exist_ok=True)

cc = OpenCC('t2s')
# simplified_text = cc.convert(full_scription)

for audio_path in audio_files:
    print(f"\nTranscribing: {audio_path.name}")
    display(Audio(str(audio_path)))

    audio_name = audio_path.stem
    lesson_folder = output_root / audio_name
    lesson_folder.mkdir(exist_ok=True)

    copied_audio = lesson_folder / 'audio'
    shutil.copy2(audio_path, copied_audio)

    result = model.transcribe(
        str(audio_path),
        language="zh",
        fp16=(device == "cuda")
    )

    timestamp_data = [
        {
            "start": segment["start"],
            "end": segment["end"],
            "text": cc.convert(segment["text"].strip())
            # "text": segment["text"].strip()

        }
        for segment in result["segments"]
    ]

    with open(
        lesson_folder / "raw_timestamp.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(timestamp_data, f, ensure_ascii=False, indent=4)

    transcript = "\n".join(item["text"] for item in timestamp_data)

    with open(
        lesson_folder / "raw_text.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(transcript)

    simplified_text = cc.convert(transcript)

    if simplified_text != transcript:
        print(f"{audio_name} audio transcribed to Traditional Chinese")

    with open(
        lesson_folder / "text.txt",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(simplified_text)

    print(transcript[:100])
    print(f"Saved to: {lesson_folder}")

zip_path = shutil.make_archive(
    "/kaggle/working/whisper_output",
    "zip",
    str(output_root)
)

print("\nFinished.")
display(FileLink(zip_path))